[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/razamoraz/FIsicaESTAdistica_2027_1/blob/main/Notebooks/Python/03_Ensamble_Microcanonico_Temperatura_Negativa.ipynb)

# 🧲 Laboratorio: Ensamble Microcanónico, Asimetría (Skewness) y Temperaturas Negativas
**Curso:** Física Estadística (2027-1) — Facultad de Ciencias, UNAM  
**Profesor:** Dr. Roberto Antonio Zamora Zamora  
**Referencias Principales:**  
- F. Reif, *Fundamentals of Statistical and Thermal Physics* (1965), Capítulos 2 y 3 (Secs. 2.1–2.5, 3.3–3.4).  
- N. F. Ramsey, *Thermodynamics and Statistical Mechanics at Negative Absolute Temperatures*, Phys. Rev. **103**, 20 (1956).  
- E. M. Purcell & R. V. Pound, *A Nuclear Spin System at Negative Temperature*, Phys. Rev. **81**, 279 (1951).  

---

## 🎯 Objetivos de la Sesión
1. **Dominar los fundamentos del Ensamble Microcanónico**: microestados accesibles $\Omega(E)$, densidad de estados $\omega(E)$ y el **Postulado Fundamental de Probabilidades *a priori* Iguales**.
2. **Analizar la interacción térmica entre subsistemas**: deducción de la probabilidad conjunta $P(E) \propto \Omega(E)\,\Omega'(E^{(0)} - E)$, definición estadística de la entropía $S = k_B \ln \Omega$, del parámetro $\beta = \frac{\partial \ln \Omega}{\partial E}$ y de la temperatura absoluta $T = 1/(k_B \beta)$.
3. **Cuantificar la agudeza (*sharpness*) y la asimetría (*skewness*) en el límite termodinámico ($N \to \infty$)**: demostrar cómo el sesgo $\gamma_1 \propto 1/\sqrt{N}$ desaparece y la distribución colapsa a una delta de Dirac gaussiana ultra-estrecha.
4. **Explorar sistemas con espectro acotado (paramagneto de espines $1/2$)**: estudiar la inversión de población, entropía cóncava y el surgimiento riguroso de **Temperaturas Absolutas Negativas ($T < 0$)**.
5. **Demostrar por qué los estados con $T < 0$ son estrictamente MÁS CALIENTES que los estados con $T > 0$**: verificar la dirección espontánea de transferencia de calor vía la Segunda Ley $\Delta S_{\text{tot}} \ge 0$ y entender por qué $\beta$ es la escala física monotónica de calor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb, gammaln
from scipy.stats import norm, skew

# Configuración estética global de gráficos
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.35
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["font.size"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["lines.linewidth"] = 2.0

print("✅ Bibliotecas científicas importadas y configuración estética establecida.")

---
## 1. El Ensamble Microcanónico y el Postulado Fundamental (Reif Cap. 2)

### 1.1 Microestados vs. Macroestados
- **Microestado (descripción cuántica):** Especificación completa de los $f$ números cuánticos $\{n_1, n_2, \dots, n_f\}$ que determinan la función de onda $\psi$ del sistema de $f$ grados de libertad.
- **Microestado (descripción clásica):** Punto en el espacio de fases $(q_1, \dots, q_f, p_1, \dots, p_f)$ contenido en una celda elemental de volumen $h_0^f = \prod_{k=1}^f (\delta q_k \delta p_k)$.
- **Macroestado:** Especificación de unos pocos parámetros macroscópicos medibles externamente (ej. energía total $E$, volumen $V$, número de partículas $N$, campo magnético $B$).

### 1.2 El Postulado Fundamental de Probabilidades *a Priori* Iguales
> **Postulado Fundamental (Reif Sec. 2.3):**  
> *Para un sistema macroscópico aislado en equilibrio termodinámico, con energía fija en el intervalo $[E, E + \delta E]$, el sistema tiene la **misma probabilidad** de encontrarse en cualquiera de sus microestados accesibles.*  
> 
> Si $\Omega(E)$ es el número de microestados accesibles en dicho rango, la probabilidad $P_r$ de que el sistema se encuentre en un microestado particular $r$ es estrictamente uniforme:
> $$ P_r = \begin{cases} \dfrac{1}{\Omega(E)}, & \text{si } E_r \in [E, E + \delta E] \\[8pt] 0, & \text{en caso contrario} \end{cases} $$

### 1.3 Comportamiento Hiper-Creciente de la Densidad de Estados $\Omega(E)$
La cantidad $\Omega(E) = \omega(E) \,\delta E$ representa el número de estados en la capa de energía $[E, E + \delta E]$, donde $\omega(E) = \frac{\partial \Phi(E)}{\partial E}$ es la **densidad de estados** y $\Phi(E)$ es el volumen total de estados con energía menor o igual a $E$.

Para un sistema macroscópico con $f \sim 10^{23}$ grados de libertad (Reif Ec. 2.5.10):
$$ \Phi(E) \propto E^{\alpha f} \implies \Omega(E) \approx \frac{\partial \Phi}{\partial E}\,\delta E \propto f E^{\alpha f - 1} \,\delta E \implies \Omega(E) \propto E^f $$
donde $\alpha \sim \mathcal{O}(1)$ (por ejemplo, $\alpha = 3/2$ para un gas ideal monoatómico en 3D).

Por consiguiente, el logaritmo de $\Omega(E)$ escala linealmente con los grados de libertad:
$$ \ln \Omega(E) \approx f \ln E + \text{constante} \sim \mathcal{O}(f) $$
lo cual demuestra matemáticamente que $\ln \Omega(E)$ es una propiedad **extensiva**.

In [ ]:
# Visualización del hiper-crecimiento de Omega(E) y la escala extensiva de ln(Omega)
E_norm = np.linspace(0.1, 3.0, 400)
degrees_of_freedom = [2, 6, 12, 30, 80]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Crecimiento explosivo de Omega(E) normalizado a E=1
for f_deg in degrees_of_freedom:
    omega_curve = (E_norm) ** (f_deg / 2.0)
    omega_curve_scaled = omega_curve / np.max(omega_curve)
    ax1.plot(E_norm, omega_curve_scaled, label=f"$f = {f_deg}$ grados de libertad")

ax1.set_title(r"Crecimiento explosivo de $\Omega(E) \propto E^{f/2}$ (escalado)")
ax1.set_xlabel(r"Energía adimensional $E / E_0$")
ax1.set_ylabel(r"$\Omega(E) / \Omega_{\max}$")
ax1.set_ylim(-0.02, 1.05)
ax1.legend()

# Panel 2: ln(Omega) vs E mostrando proporcionalidad extensiva
for f_deg in degrees_of_freedom:
    log_omega = (f_deg / 2.0) * np.log(E_norm)
    ax2.plot(E_norm, log_omega, label=f"$f = {f_deg}$")

ax2.set_title(r"Comportamiento suave y extensivo de $\ln \Omega(E) = \frac{f}{2} \ln E$")
ax2.set_xlabel(r"Energía adimensional $E / E_0$")
ax2.set_ylabel(r"$\ln \Omega(E)$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 2. Interacción Térmica y Distribución de Energía $P(E)$ (Reif Cap. 3)

Consideremos dos subsistemas macroscópicos $A$ y $A'$ en **contacto térmico débil** (es decir, pueden intercambiar energía en forma de calor, pero sus parámetros externos como el volumen permanecen fijos):
- El sistema compuesto $A^{(0)} = A + A'$ está **totalmente aislado** del exterior, por lo que la energía total se conserva:
  $$ E + E' = E^{(0)} = \text{constante} \implies E' = E^{(0)} - E $$
- El número total de microestados del sistema combinado cuando $A$ tiene energía $E$ es simplemente el producto de multiplicidades (Reif Ec. 3.3.4):
  $$ \Omega^{(0)}(E) = \Omega(E) \,\Omega'(E') = \Omega(E) \,\Omega'(E^{(0)} - E) $$
- Aplicando el postulado fundamental al sistema global $A^{(0)}$, la probabilidad $P(E)$ de encontrar al subsistema $A$ con energía $E$ es (Reif Ec. 3.3.5):
  $$ P(E) = C \,\Omega(E) \,\Omega'(E^{(0)} - E) = \frac{\Omega(E) \,\Omega'(E^{(0)} - E)}{\sum_E \Omega(E) \,\Omega'(E^{(0)} - E)} $$

### 2.1 La Condición de Equilibrio Térmico y el Nacimiento de la Temperatura
Para encontrar la energía más probable $\tilde{E}$ (el máximo de $P(E)$ o, equivalentemente, de $\ln P(E)$):
$$ \frac{\partial \ln P(E)}{\partial E} = 0 \implies \frac{\partial \ln \Omega(E)}{\partial E} + \frac{\partial \ln \Omega'(E')}{\partial E'} \underbrace{\frac{\partial E'}{\partial E}}_{-1} = 0 $$

Definiendo el **parámetro fundamental $\beta$** (Reif Ec. 3.3.9):
$$ \beta(E) \equiv \frac{\partial \ln \Omega(E)}{\partial E} $$

La condición del macroestado más probable (equilibrio térmico) es la **igualdad de los parámetros $\beta$**:
$$ \beta(\tilde{E}) = \beta'(\tilde{E}') $$

### 2.2 Entropía de Boltzmann y Temperatura Absoluta
Definiendo la **Entropía de Boltzmann** (Reif Ec. 3.3.12):
$$ S(E) \equiv k_B \ln \Omega(E) $$
y la **Temperatura Absoluta $T$**:
$$ \frac{1}{T} \equiv \frac{\partial S}{\partial E} = k_B \beta \implies T = \frac{1}{k_B \beta} $$

La condición de equilibrio térmico $\beta = \beta'$ equivale idénticamente a:
$$ T = T' \quad \text{o} \quad S_{\text{tot}}(E) = S(E) + S'(E^{(0)} - E) = \text{MÁXIMO} $$

In [ ]:
# Demostración del producto Omega(E) * Omega'(E_tot - E) y el surgimiento del pico agudo P(E)
E_tot = 100.0
E_A = np.linspace(0.5, E_tot - 0.5, 500)
E_B = E_tot - E_A

# Grados de libertad moderados para ilustrar las curvas
f_A = 16
f_B = 24

# Calculamos log-multiplicidades: ln(Omega) = (f/2) * ln(E)
log_omega_A = (f_A / 2.0) * np.log(E_A)
log_omega_B = (f_B / 2.0) * np.log(E_B)
log_P = log_omega_A + log_omega_B

# Probabilidad normalizada P(E_A)
P_unnorm = np.exp(log_P - np.max(log_P))  # estabilidad numérica
P_norm = P_unnorm / np.trapezoid(P_unnorm, E_A)

# Energía teórica del máximo:
E_A_tilde = E_tot * f_A / (f_A + f_B)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Curvas individuales de multiplicidad
ax1.plot(E_A, np.exp(log_omega_A - np.max(log_omega_A)), "b-", label=r"$\Omega_A(E_A) \propto E_A^{f_A/2}$ (crece)")
ax1.plot(E_A, np.exp(log_omega_B - np.max(log_omega_B)), "r--", label=r"$\Omega_B(E_{\mathrm{tot}} - E_A) \propto (E_{\mathrm{tot}}-E_A)^{f_B/2}$ (decrece)")
ax1.set_title("Factores en Competencia de Multiplicidad")
ax1.set_xlabel(r"Energía del subsistema $A$ ($E_A$)")
ax1.set_ylabel("Multiplicidad relativa (normalizada a 1)")
ax1.legend()

# Panel 2: Probabilidad conjunta resultante P(E_A)
ax2.plot(E_A, P_norm, "purple", linewidth=2.5, label=r"$P(E_A) = C \, \Omega_A(E_A)\,\Omega_B(E_{\mathrm{tot}}-E_A)$")
ax2.axvline(E_A_tilde, color="darkgreen", linestyle=":", linewidth=2, label=f"Máximo teórico $\\tilde{{E}}_A = {E_A_tilde:.1f}$")
ax2.fill_between(E_A, P_norm, color="purple", alpha=0.2)
ax2.set_title(r"Distribución de Probabilidad Conjunta $P(E_A)$")
ax2.set_xlabel(r"Energía del subsistema $A$ ($E_A$)")
ax2.set_ylabel(r"Densidad de probabilidad $P(E_A)$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 3. Agudeza (*Sharpness*), Asimetría (*Skewness*) y el Límite Termodinámico ($N \to \infty$)

¿Qué tan ancho es el pico de $P(E)$ y qué forma tiene en función del tamaño del sistema $N$?

Realicemos un desarrollo de Taylor de $\ln P(E)$ alrededor de su punto máximo $\tilde{E}$:
$$ \ln P(E) = \ln P(\tilde{E}) + \underbrace{\left.\frac{\partial \ln P}{\partial E}\right|_{\tilde{E}}}_{= 0} (E - \tilde{E}) + \frac{1}{2} \left.\frac{\partial^2 \ln P}{\partial E^2}\right|_{\tilde{E}} (E - \tilde{E})^2 + \frac{1}{6} \left.\frac{\partial^3 \ln P}{\partial E^3}\right|_{\tilde{E}} (E - \tilde{E})^3 + \dots $$

### 3.1 Varianza y Agudeza (*Sharpness*)
Definiendo la curvatura de segundo orden:
$$ -\frac{1}{\sigma_E^2} = \left.\frac{\partial^2 \ln P}{\partial E^2}\right|_{\tilde{E}} = \left.\frac{\partial \beta}{\partial E}\right|_{\tilde{E}} + \left.\frac{\partial \beta'}{\partial E'}\right|_{\tilde{E}'} $$

Como $\beta \sim \mathcal{O}(1)$ y $E \sim \mathcal{O}(N)$, se tiene que $\frac{\partial \beta}{\partial E} \sim -\frac{1}{N E_0^2} < 0$. Por lo tanto:
$$ \sigma_E^2 \sim \mathcal{O}(N) \implies \sigma_E \sim \mathcal{O}(\sqrt{N}) $$

> **Ley de Fluctuaciones Relativas (Reif Sec. 3.3):**  
> La fluctuación relativa de la energía respecto a la media escala como:
> $$ \frac{\sigma_E}{\tilde{E}} \sim \frac{\sqrt{N}}{N} = \frac{1}{\sqrt{N}} \xrightarrow{N \to \infty} 0 $$
> Para $N \sim 10^{24}$, $\frac{\sigma_E}{\tilde{E}} \sim 10^{-12}$, lo que significa que el pico es **infinitamente agudo** y el macroestado observado es indistinguible de una delta de Dirac $\delta(E - \tilde{E})$.

### 3.2 Asimetría (*Skewness*) $\gamma_1$ en Sistemas Finitos vs. Límite Termodinámico
El tercer término del desarrollo de Taylor gobierna la **asimetría** (*skewness*) de la distribución:
$$ \kappa_3 = \left.\frac{\partial^3 \ln P}{\partial E^3}\right|_{\tilde{E}} \sim \mathcal{O}\left(\frac{1}{N^2}\right) $$

El coeficiente de asimetría estandarizado $\gamma_1 = \frac{\mu_3}{\sigma_E^3} = \frac{\langle (E - \bar{E})^3 \rangle}{\sigma_E^3}$ escala como:
$$ \gamma_1 \approx \sigma_E^3 \,\kappa_3 \sim (N^{1/2})^3 \cdot \frac{1}{N^2} = \frac{N^{3/2}}{N^2} = \frac{1}{\sqrt{N}} \xrightarrow{N \to \infty} 0 $$

- En **sistemas finitos pequeños ($N \sim 4 - 50$)**, $P(E)$ presenta un sesgo (*skewness*) visible hacia la derecha o izquierda porque el crecimiento potencial $\Omega(E) \propto E^{f/2}$ introduce una asimetría intrínseca.
- En el **límite termodinámico ($N \to \infty$)**, el sesgo $\gamma_1 \to 0$, la distribución se vuelve rigurosamente simétrica y colapsa a la campana de Gauss perfecta $\mathcal{N}(\tilde{E}, \sigma_E^2)$.

In [ ]:
# Estudio numérico de la agudeza (Sharpness) y asimetría (Skewness) en función de N
N_particles_list = [6, 20, 80, 400, 2000]
E_per_particle = 2.0  # energía promedio por partícula

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

skewness_vals = []
relative_width_vals = []

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_particles_list)))

for idx, N in enumerate(N_particles_list):
    f_A = 3 * N  # 3 grados de libertad por partícula en 3D
    f_B = 3 * N
    E_tot = 2.0 * N * E_per_particle
    
    E_tilde = E_tot / 2.0
    sigma_est = E_tilde / np.sqrt(f_A / 2.0)
    E_grid = np.linspace(max(0.01, E_tilde - 4.5 * sigma_est), min(E_tot - 0.01, E_tilde + 4.5 * sigma_est), 1000)
    
    log_P = (f_A / 2.0) * np.log(E_grid) + (f_B / 2.0) * np.log(E_tot - E_grid)
    P = np.exp(log_P - np.max(log_P))
    P_norm = P / np.trapezoid(P, E_grid)
    
    mean_E = np.trapezoid(E_grid * P_norm, E_grid)
    var_E = np.trapezoid(((E_grid - mean_E) ** 2) * P_norm, E_grid)
    sigma_num = np.sqrt(var_E)
    mu3_num = np.trapezoid(((E_grid - mean_E) ** 3) * P_norm, E_grid)
    skew_num = mu3_num / (sigma_num ** 3)
    
    skewness_vals.append(abs(skew_num))
    relative_width_vals.append(sigma_num / mean_E)
    
    # Variable estandarizada z = (E - mean) / sigma para verificar colapso a Gaussiana N(0,1)
    z_grid = (E_grid - mean_E) / sigma_num
    P_z = P_norm * sigma_num  # densidad de probabilidad estandarizada
    
    ax1.plot(z_grid, P_z, label=f"$N = {N}$ ($\\gamma_1 = {skew_num:.3f}$)", color=colors[idx])

# Referencia normal estándar N(0, 1)
z_ref = np.linspace(-4, 4, 300)
ax1.plot(z_ref, norm.pdf(z_ref), "k--", linewidth=2.5, label=r"Gaussiana estándar $\mathcal{N}(0,1)$")
ax1.set_title(r"Colapso a la Gaussiana y Desaparición de Skewness ($z = \frac{E - \bar{E}}{\sigma_E}$)")
ax1.set_xlabel(r"Variable tipificada $z$")
ax1.set_ylabel(r"Densidad de probabilidad $P(z)$")
ax1.set_xlim(-3.5, 3.5)
ax1.legend()

# Panel 2: Escalamiento del Skewness y Fluctuación Relativa vs N en escala log-log
N_arr = np.array(N_particles_list)
ax2.loglog(N_arr, relative_width_vals, "o-", color="crimson", label=r"Fluctuación Relativa $\sigma_E / \bar{E} \propto N^{-1/2}$")
ax2.loglog(N_arr, skewness_vals, "s--", color="teal", label=r"Asimetría / Skewness $|\gamma_1| \propto N^{-1/2}$")
ax2.loglog(N_arr, 1.0 / np.sqrt(N_arr), "k:", alpha=0.7, label=r"Pendiente teórica $N^{-1/2}$")
ax2.set_title(r"Convergencia al Límite Termodinámico ($1/\sqrt{N} \to 0$)")
ax2.set_xlabel(r"Número de partículas $N$")
ax2.set_ylabel(r"Magnitud relativa")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 4. El Sistema de Espines $1/2$ en Campo Magnético y el Espectro Acotado

Consideremos un sistema de $N$ partículas fijas de **espín $1/2$**, cada una con momento magnético dipolar $\mu$, sumergidas en un campo magnético uniforme $B$ apuntando en la dirección $+z$ (Reif Sec. 2.1, Ejemplo 2 y Sec. 2.2):

- Cada espín individual tiene solo dos microestados cuánticos posibles:
  - **Espín Arriba ($\uparrow$, paralelo a $B$):** momento magnético $+\mu$, energía $\epsilon_+ = -\mu B$.
  - **Espín Abajo ($\downarrow$, antiparalelo a $B$):** momento magnético $-\mu$, energía $\epsilon_- = +\mu B$.

Si el sistema tiene $N_+$ espines en $\uparrow$ y $N_- = N - N_+$ espines en $\downarrow$, la **energía total** es:
$$ E = N_+ (-\mu B) + N_- (+\mu B) = -(N_+ - N_-) \mu B = -(2N_+ - N) \mu B $$

Despejando el número de espines arriba y abajo en términos de $E$:
$$ N_+ = \frac{N - E/(\mu B)}{2}, \qquad N_- = \frac{N + E/(\mu B)}{2} $$

### 4.1 Característica Clave: El Espectro de Energía es Estrictamente Acotado
A diferencia de un gas clásico (cuya energía cinética $E = \sum p_i^2/2m \in [0, \infty)$ puede ser arbitrariamente grande), el sistema de espines posee **cotas finitas inferior y superior**:
$$ E_{\min} = -N\mu B \quad (\text{todos } \uparrow) \le E \le +N\mu B = E_{\max} \quad (\text{todos } \downarrow) $$

### 4.2 Número Exacto de Microestados Accesibles $\Omega(E)$
La multiplicidad microcanónica es el número de combinaciones de $N$ espines tomados de $N_+$ en $N_+$:
$$ \Omega(E) = \binom{N}{N_+} = \frac{N!}{N_+! (N - N_+)!} = \frac{N!}{\left(\frac{N - E/(\mu B)}{2}\right)! \left(\frac{N + E/(\mu B)}{2}\right)!} $$

> **Propiedad Fundamental de $\Omega(E)$:**  
> - $\Omega(E)$ es una función **simétrica** respecto a $E = 0$.
> - Para $E = -N\mu B$: $\Omega = \binom{N}{N} = 1$ (microestado único fundamental).
> - En $E = 0$: $N_+ = N_- = N/2$, alcanzando su **máximo absoluto** $\Omega_{\max} = \binom{N}{N/2}$.
> - Para $E > 0$: ¡$\Omega(E)$ **decrece** con el incremento de la energía $E$!
> - Para $E = +N\mu B$: $\Omega = \binom{N}{0} = 1$ (microestado único de máxima energía).

In [ ]:
# Multiplicidad exacta Omega(E) y Entropía S(E) para un ensamble de espines 1/2
N_spins = 100
mu_B_field = 1.0  # unidades de mu * B

# Valores posibles de N+ discretos
N_plus_arr = np.arange(0, N_spins + 1)
N_minus_arr = N_spins - N_plus_arr
E_spins = -(2 * N_plus_arr - N_spins) * mu_B_field

# log(Omega) numéricamente exacto usando gammaln
log_omega_spins = gammaln(N_spins + 1) - gammaln(N_plus_arr + 1) - gammaln(N_minus_arr + 1)
S_spins = log_omega_spins  # en unidades de k_B

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Multiplicidad Omega(E)
ax1.plot(E_spins, np.exp(log_omega_spins - np.max(log_omega_spins)), "b-o", markersize=4, label=r"$\Omega(E) = \binom{N}{N_+}$")
ax1.axvline(0, color="red", linestyle="--", alpha=0.7, label=r"Máximo en $E = 0$ ($N_+ = N_- = N/2$)")
ax1.set_title(f"Multiplicidad de Microestados $\\Omega(E)$ ($N = {N_spins}$ espines)")
ax1.set_xlabel(r"Energía total $E / (\mu B)$")
ax1.set_ylabel(r"$\Omega(E) / \Omega_{\max}$")
ax1.legend()

# Panel 2: Entropía S(E) / (N k_B)
ax2.plot(E_spins / N_spins, S_spins / N_spins, color="teal", linewidth=2.5, label=r"$S(E)/(N k_B) = \frac{1}{N}\ln\Omega(E)$")
ax2.axhline(np.log(2), color="gray", linestyle=":", label=r"$S_{\max}/(Nk_B) = \ln(2) \approx 0.693$")
ax2.axvline(0, color="red", linestyle="--", alpha=0.7)
ax2.annotate("Pendiente positiva:\n" + r"$dS/dE > 0 \rightarrow T > 0$", xy=(-0.5, 0.45), xytext=(-0.85, 0.2),
             arrowprops=dict(arrowstyle="->", color="blue", lw=1.5), fontsize=10, color="blue")
ax2.annotate("Pendiente negativa:\n" + r"$dS/dE < 0 \rightarrow T < 0$", xy=(0.5, 0.45), xytext=(0.3, 0.2),
             arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5), fontsize=10, color="crimson")
ax2.set_title(r"Entropía por Espín $S(E)/(N k_B)$ vs. Energía Adimensional $E/(N \mu B)$")
ax2.set_xlabel(r"Energía específica $\varepsilon = E / (N \mu B)$")
ax2.set_ylabel(r"$S / (N k_B)$")
ax2.set_ylim(-0.02, 0.78)
ax2.legend(loc="upper center")

plt.tight_layout()
plt.show()

---
## 5. Termodinámica de Espines: Parámetro $\beta(E)$ y Temperaturas Negativas ($T < 0$)

Aplicando la aproximación de Stirling $\ln(n!) \approx n \ln n - n$ a la entropía para $N \gg 1$ y definiendo la variable adimensional $\varepsilon \equiv \frac{E}{N\mu B} \in [-1, 1]$:
$$ \frac{S(\varepsilon)}{N k_B} = \ln 2 - \frac{1 + \varepsilon}{2}\ln(1 + \varepsilon) - \frac{1 - \varepsilon}{2}\ln(1 - \varepsilon) $$

### 5.1 Parámetro Inverso $\beta(E)$
Diferenciando respecto a la energía $E = N\mu B \,\varepsilon$:
$$ \beta(E) = \frac{1}{k_B}\frac{\partial S}{\partial E} = \frac{1}{2\mu B} \ln\left( \frac{1 - \varepsilon}{1 + \varepsilon} \right) = \frac{1}{2\mu B} \ln\left( \frac{N_+}{N_-} \right) $$

### 5.2 Temperatura Absoluta $T(E)$
$$ T(E) = \frac{1}{k_B \beta(E)} = \frac{2\mu B / k_B}{\ln\left( \frac{1 - \varepsilon}{1 + \varepsilon} \right)} = \frac{2\mu B / k_B}{\ln(N_+ / N_-)} $$

| Régimen de Energía | Razón de Poblaciones | Pendiente $\frac{\partial S}{\partial E}$ | Parámetro $\beta$ | Temperatura Absoluta $T$ | Estado Físico |
| :---: | :---: | :---: | :---: | :---: | :---: |
| $\varepsilon = -1$ ($E = -N\mu B$) | $N_+ = N, N_- = 0$ | $+\infty$ | $+\infty$ | $T = +0\,\text{K}$ | **Estado fundamental** (orden total en $\uparrow$) |
| $-1 < \varepsilon < 0$ ($E < 0$) | $N_+ > N_-$ | $> 0$ | $\beta > 0$ | $T > 0$ | **Régimen térmico usual** (mayoría en fundamental) |
| $\varepsilon = 0$ ($E = 0$) | $N_+ = N_- = N/2$ | $0$ | $\beta = 0$ | $T = \pm \infty$ | **Desorden máximo** ($S = S_{\max}$, $M = 0$) |
| $0 < \varepsilon < +1$ ($E > 0$) | $N_- > N_+$ | $< 0$ | $\beta < 0$ | $T < 0$ | **Inversión de Población** (mayoría en excitado) |
| $\varepsilon = +1$ ($E = +N\mu B$) | $N_+ = 0, N_- = N$ | $-\infty$ | $-\infty$ | $T = -0\,\text{K}$ | **Estado de máxima energía** (orden total en $\downarrow$) |

In [ ]:
# Gráficas de los 4 pilares termodinámicos: S(E), beta(E), T(E) y Poblaciones N+/N, N-/N
eps = np.linspace(-0.96, 0.96, 500)

# 1. Entropía por partícula
s_per_particle = np.log(2) - 0.5 * (1 + eps) * np.log(1 + eps) - 0.5 * (1 - eps) * np.log(1 - eps)

# 2. Beta (en unidades de 1 / (mu B))
beta_arr = 0.5 * np.log((1 - eps) / (1 + eps))

# 3. Temperatura T (en unidades de mu B / k_B)
T_arr = 1.0 / beta_arr

# 4. Fracciones de población
frac_plus = (1 - eps) / 2.0   # espines arriba
frac_minus = (1 + eps) / 2.0  # espines abajo

fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# (0,0) Entropía
axs[0, 0].plot(eps, s_per_particle, "darkblue", linewidth=2.5)
axs[0, 0].axvline(0, color="red", linestyle="--", alpha=0.6)
axs[0, 0].set_title(r"(a) Entropía $S / (N k_B)$ vs. Energía $\varepsilon$")
axs[0, 0].set_xlabel(r"Energía adimensional $\varepsilon = E / (N \mu B)$")
axs[0, 0].set_ylabel(r"$S / (N k_B)$")

# (0,1) Parámetro beta(E)
axs[0, 1].plot(eps, beta_arr, "purple", linewidth=2.5)
axs[0, 1].axhline(0, color="black", linestyle="-", alpha=0.4)
axs[0, 1].axvline(0, color="red", linestyle="--", alpha=0.6)
axs[0, 1].fill_between(eps, beta_arr, 0, where=(beta_arr > 0), color="blue", alpha=0.15, label=r"$\beta > 0$ ($T > 0$)")
axs[0, 1].fill_between(eps, beta_arr, 0, where=(beta_arr < 0), color="crimson", alpha=0.15, label=r"$\beta < 0$ ($T < 0$)")
axs[0, 1].set_title(r"(b) Parámetro Inverso $\beta \,\mu B = \frac{1}{2} \ln\frac{1-\varepsilon}{1+\varepsilon}$")
axs[0, 1].set_xlabel(r"Energía adimensional $\varepsilon = E / (N \mu B)$")
axs[0, 1].set_ylabel(r"$\beta \,\mu B$")
axs[0, 1].legend()

# (1,0) Temperatura T(E)
mask_pos = eps < -0.01
mask_neg = eps > 0.01
axs[1, 0].plot(eps[mask_pos], T_arr[mask_pos], "blue", linewidth=2.5, label=r"$T > 0$ (Régimen usual)")
axs[1, 0].plot(eps[mask_neg], T_arr[mask_neg], "crimson", linewidth=2.5, label=r"$T < 0$ (Inversión poblacional)")
axs[1, 0].axvline(0, color="red", linestyle="--", alpha=0.6, label=r"Discontinuidad: $T \to \pm \infty$")
axs[1, 0].set_title(r"(c) Temperatura Absoluta $k_B T / (\mu B)$ vs. $\varepsilon$")
axs[1, 0].set_xlabel(r"Energía adimensional $\varepsilon = E / (N \mu B)$")
axs[1, 0].set_ylabel(r"$k_B T / (\mu B)$")
axs[1, 0].set_ylim(-12, 12)
axs[1, 0].legend()

# (1,1) Inversión de poblaciones N+ y N-
axs[1, 1].plot(eps, frac_plus, "blue", linewidth=2.2, label=r"$N_+ / N$ (espines arriba, estado base)")
axs[1, 1].plot(eps, frac_minus, "crimson", linewidth=2.2, label=r"$N_- / N$ (espines abajo, estado excitado)")
axs[1, 1].axvline(0, color="red", linestyle="--", alpha=0.6)
axs[1, 1].axhline(0.5, color="gray", linestyle=":")
axs[1, 1].annotate("Inversión de Población:\n$N_- > N_+$", xy=(0.5, 0.75), xytext=(0.2, 0.85),
                   arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5), fontsize=10, color="crimson")
axs[1, 1].set_title(r"(d) Fracciones de Población ($N_+/N$ vs. $N_-/N$)")
axs[1, 1].set_xlabel(r"Energía adimensional $\varepsilon = E / (N \mu B)$")
axs[1, 1].set_ylabel("Fracción de espines")
axs[1, 1].legend()

plt.tight_layout()
plt.show()

---
## 6. Demostración Rigurosa: ¿Por qué los Estados con $T < 0$ son MÁS CALIENTES que los Estados con $T > 0$?

Uno de los resultados más fascinantes y anti-intuitivos de la física estadística es que **un cuerpo a temperatura absoluta negativa es estrictamente más caliente que cualquier cuerpo a temperatura positiva** (incluso más caliente que $T = +\infty$).

### 6.1 Demostración mediante la Segunda Ley de la Termodinámica
Consideremos dos sistemas $1$ y $2$ que se ponen en **contacto térmico**:
- El sistema 1 tiene temperatura inicial $T_1$ (parámetro $\beta_1 = 1/k_B T_1$).
- El sistema 2 tiene temperatura inicial $T_2$ (parámetro $\beta_2 = 1/k_B T_2$).
- Conservación de la energía: el calor cedido por uno es absorbido por el otro ($Q = \Delta E_1 = -\Delta E_2$).

El cambio total de entropía en el proceso es:
$$ \Delta S_{\text{tot}} = \Delta S_1 + \Delta S_2 \approx \left(\frac{\partial S_1}{\partial E_1}\right)\Delta E_1 + \left(\frac{\partial S_2}{\partial E_2}\right)\Delta E_2 = k_B \beta_1 \Delta E_1 + k_B \beta_2 (-\Delta E_1) $$
$$ \Delta S_{\text{tot}} = k_B (\beta_1 - \beta_2) Q \ge 0 $$

### 6.2 Análisis de los Signos
Supongamos que el **Sistema 1 está a temperatura negativa ($T_1 < 0 \implies \beta_1 < 0$)** y el **Sistema 2 está a temperatura positiva ($T_2 > 0 \implies \beta_2 > 0$)**:
$$ \beta_1 < 0 < \beta_2 \implies (\beta_1 - \beta_2) < 0 $$

Para que la Segunda Ley de la Termodinámica se cumpla ($\Delta S_{\text{tot}} \ge 0$), el producto $(\beta_1 - \beta_2) Q$ debe ser positivo:
$$ \underbrace{(\beta_1 - \beta_2)}_{< 0} \, Q \ge 0 \implies Q \le 0 $$

> **Conclusión Física:**  
> Como $Q = \Delta E_1 \le 0$, el Sistema 1 **necesariamente cede energía en forma de calor** y el Sistema 2 **absorbe calor** ($\Delta E_2 = -Q \ge 0$).  
> 
> Por definición física: *El cuerpo que cede calor de forma espontánea es el más caliente, y el que absorbe calor es el más frío*.  
> Por lo tanto: **¡Cualquier estado con $T < 0$ es más caliente que cualquier estado con $T > 0$!**

### 6.3 La Escala Monotónica de Calor: El Parámetro $\beta = 1/(k_B T)$
La escala usual $T$ presenta una singularidad artificial ($+\infty \to -\infty$) al cruzar $\beta = 0$. La escala fundamental y monótona de "frialdad $\to$ calor" es el parámetro $\beta$:

$$ \begin{matrix}
\beta = +\infty & \longrightarrow & \beta > 0 & \longrightarrow & \beta = 0 & \longrightarrow & \beta < 0 & \longrightarrow & \beta = -\infty \\[6pt]
T = +0\,\text{K} & & T > 0 & & T = \pm\infty & & T < 0 & & T = -0\,\text{K} \\[6pt]
\textbf{CERO ABSOLUTO} & & \text{Tibio / Caliente} & & \text{Muy caliente} & & \text{Ultra caliente} & & \textbf{ESTADO MÁS CALIENTE} \\[2pt]
(\text{Más Frío}) & & & & & & & & (\text{Máxima Energía})
\end{matrix} $$

In [ ]:
# Simulación de Contacto Térmico: Sistema con T < 0 en contacto con Sistema con T > 0
# Sistema A: N_A = 200 espines, preparado a T_A < 0 (alta energía E_A > 0)
# Sistema B: N_B = 300 espines, preparado a T_B > 0 (baja energía E_B < 0)

N_A = 200
N_B = 300
muB = 1.0

# Estados iniciales:
eps_A_initial = +0.50  # E_A = +100 muB (T_A < 0)
eps_B_initial = -0.30  # E_B = -90 muB  (T_B > 0)

E_A_init = eps_A_initial * N_A * muB
E_B_init = eps_B_initial * N_B * muB
E_total = E_A_init + E_B_init  # E_total = +100 - 90 = +10 muB

beta_A_init = 0.5 / muB * np.log((1 - eps_A_initial) / (1 + eps_A_initial))
beta_B_init = 0.5 / muB * np.log((1 - eps_B_initial) / (1 + eps_B_initial))
T_A_init = 1.0 / beta_A_init
T_B_init = 1.0 / beta_B_init

print("=== ESTADO INICIAL ANTES DEL CONTACTO TÉRMICO ===")
print(f"Sistema A (T < 0): E_A = {E_A_init:+6.1f} muB | beta_A = {beta_A_init:+6.3f} | T_A = {T_A_init:+6.2f} muB/kB (INVERSIÓN POBLACIONAL)")
print(f"Sistema B (T > 0): E_B = {E_B_init:+6.1f} muB | beta_B = {beta_B_init:+6.3f} | T_B = {T_B_init:+6.2f} muB/kB (RÉGIMEN ORDINARIO)")
print(f"Diferencia beta_A - beta_B = {beta_A_init - beta_B_init:+.3f} < 0 (Sistema A es MÁS CALIENTE)")

# Rango de energías posibles para el subsistema A en contacto térmico
E_A_range = np.linspace(max(-N_A * muB + 1, E_total - N_B * muB + 1), 
                        min(N_A * muB - 1, E_total + N_B * muB - 1), 600)
E_B_range = E_total - E_A_range

# Multiplicidad conjunta ln Omega^(0)(E_A) = ln Omega_A(E_A) + ln Omega_B(E_total - E_A)
N_A_plus = (N_A - E_A_range / muB) / 2.0
N_A_minus = (N_A + E_A_range / muB) / 2.0
N_B_plus = (N_B - E_B_range / muB) / 2.0
N_B_minus = (N_B + E_B_range / muB) / 2.0

log_Omega_total = (gammaln(N_A + 1) - gammaln(N_A_plus + 1) - gammaln(N_A_minus + 1) +
                   gammaln(N_B + 1) - gammaln(N_B_plus + 1) - gammaln(N_B_minus + 1))

# Punto de equilibrio (máximo de probabilidad / entropía conjunta)
idx_eq = np.argmax(log_Omega_total)
E_A_eq = E_A_range[idx_eq]
E_B_eq = E_total - E_A_eq

eps_A_eq = E_A_eq / (N_A * muB)
eps_B_eq = E_B_eq / (N_B * muB)
beta_final = 0.5 / muB * np.log((1 - eps_A_eq) / (1 + eps_A_eq))
T_final = 1.0 / beta_final

Q_transfer = E_A_eq - E_A_init  # Calor cedido por A

print("\n=== ESTADO FINAL DE EQUILIBRIO TÉRMICO ===")
print(f"Sistema A (final): E_A* = {E_A_eq:+6.1f} muB (Cambio: Delta E_A = {Q_transfer:+.1f} muB, CEDIÓN DE CALOR)")
print(f"Sistema B (final): E_B* = {E_B_eq:+6.1f} muB (Cambio: Delta E_B = {-Q_transfer:+.1f} muB, ABSORCIÓN DE CALOR)")
print(f"Temperatura de equilibrio común: beta_eq = {beta_final:+.4f} | T_eq = {T_final:+.2f} muB/kB")

# Gráfica de la transferencia de calor y maximización de la entropía
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Distribución de probabilidad del estado combinado P(E_A)
P_comb = np.exp(log_Omega_total - np.max(log_Omega_total))
P_comb_norm = P_comb / np.trapezoid(P_comb, E_A_range)
ax1.plot(E_A_range, P_comb_norm, color="darkviolet", linewidth=2.5, label=r"$P(E_A) \propto \Omega_A(E_A)\Omega_B(E_{\mathrm{tot}}-E_A)$")
ax1.axvline(E_A_init, color="crimson", linestyle="--", linewidth=2, label=f"Estado inicial $E_A^{{(i)}} = {E_A_init:+.0f}$ ($T_A < 0$)")
ax1.axvline(E_A_eq, color="darkgreen", linestyle="-", linewidth=2.5, label=f"Equilibrio final $E_A^* = {E_A_eq:+.1f}$ ($T_f = {T_final:+.1f}$)")
ax1.annotate("", xy=(E_A_eq, np.max(P_comb_norm)*0.5), xytext=(E_A_init, np.max(P_comb_norm)*0.5),
             arrowprops=dict(arrowstyle="->", color="red", lw=2.5))
ax1.text((E_A_init + E_A_eq)/2, np.max(P_comb_norm)*0.55, f"Calor transferido\n$Q = {Q_transfer:.1f} \\mu B$", 
         ha="center", color="red", fontweight="bold")
ax1.set_title("Evolución Espontánea de la Energía en Contacto Térmico")
ax1.set_xlabel(r"Energía del subsistema $A$ ($E_A / \mu B$)")
ax1.set_ylabel(r"Densidad de probabilidad $P(E_A)$")
ax1.legend()

# Panel 2: Curvas beta_A(E_A) y beta_B(E_total - E_A) y su intersección
beta_A_curve = 0.5 / muB * np.log((1 - E_A_range/(N_A*muB)) / (1 + E_A_range/(N_A*muB)))
beta_B_curve = 0.5 / muB * np.log((1 - E_B_range/(N_B*muB)) / (1 + E_B_range/(N_B*muB)))
ax2.plot(E_A_range, beta_A_curve, "b-", label=r"$\beta_A(E_A)$")
ax2.plot(E_A_range, beta_B_curve, "r--", label=r"$\beta_B(E_{\mathrm{tot}} - E_A)$")
label_eq = f"Equilibrio: $\\beta_A = \\beta_B = {beta_final:.3f}$"
ax2.plot(E_A_eq, beta_final, "go", markersize=9, label=label_eq)
ax2.axhline(0, color="black", linestyle=":", alpha=0.5)
ax2.set_title(r"Igualación de Parámetros $\beta$: Condición de Equilibrio Térmico")
ax2.set_xlabel(r"Energía del subsistema $A$ ($E_A / \mu B$)")
ax2.set_ylabel(r"Parámetro $\beta \,\mu B$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 7. ¿Por qué los Gases Ideales y Sólidos Ordinarios NO tienen $T < 0$?

En sistemas físicos estándar formados por partículas libres o interactuantes (ej. gas ideal monoatómico en 3D):
$$ E = \sum_{i=1}^N \frac{\mathbf{p}_i^2}{2m} + U(\mathbf{r}_1, \dots, \mathbf{r}_N) \in [0, \infty) $$

Como la energía cinética traslacional $\frac{p^2}{2m}$ no tiene límite superior, el volumen del espacio de fases en el espacio de momentos crece indefinidamente con el radio $R(E) = \sqrt{2mE}$ (Reif Sec. 2.5):
$$ \Omega_{\text{gas}}(E) \propto V^N E^{3N/2} \implies S(E) = k_B \ln \Omega = k_B \left[ N \ln V + \frac{3N}{2}\ln E + \text{cte} \right] $$

Calculando el parámetro $\beta$:
$$ \beta = \frac{1}{k_B}\frac{\partial S}{\partial E} = \frac{3N}{2E} > 0 \quad \text{para todo } E > 0 $$

Por consiguiente:
$$ T = \frac{2E}{3N k_B} > 0 \quad \text{estrictamente positivo en todo el rango de energías.} $$

### 7.1 Criterios de Ramsey (1956) para la Existencia de Temperaturas Negativas
Norman Ramsey estableció en 1956 las **tres condiciones indispensables** para que un sistema físico admita estados con temperatura absoluta negativa ($T < 0$):
1. **Espectro de Energía Acotado Superiormente:** Los microestados del sistema deben poseer una cota superior finita de energía $E_{\max} < \infty$. Esto impide que $\Omega(E)$ crezca indefinidamente.
2. **Termalización Interna Rápida:** El sistema debe alcanzar el equilibrio termodinámico interno entre sus propios grados de libertad en un tiempo característico $\tau_{\text{int}}$ muy corto.
3. **Aislamiento Térmico Frente a Grados de Libertad No Acotados:** El tiempo de interacción con grados de libertad no acotados (como las vibraciones de la red cristalina o fonones, que siempre tienen $T > 0$) debe ser mucho mayor que $\tau_{\text{int}}$ (es decir, $\tau_{\text{relajación}} \gg \tau_{\text{int}}$).

Sistemas experimentales reales que satisfacen estas condiciones incluyen **espines nucleares en redes cristalinas aisladas** (Purcell & Pound, 1951) y **gases cuánticos de átomos ultrafríos en redes ópticas** (Braun et al., *Science* 339, 2013).

---
## 8. Actividades Prácticas y Retos Computacionales

### 🧠 Reto 1: Capacidad Calorífica y la Anomalía de Schottky
La capacidad calorífica a campo constante se define como $C_B = \frac{\partial E}{\partial T} = \frac{\partial E}{\partial \beta}\frac{\partial \beta}{\partial T} = -k_B \beta^2 \frac{\partial E}{\partial \beta}$.

Dado que $\varepsilon = -\tanh(\beta \mu B) \implies E = -N\mu B \tanh\left(\frac{\mu B}{k_B T}\right)$, calculemos su derivada analítica:
$$ C_B(T) = N k_B \left( \frac{\mu B}{k_B T} \right)^2 \operatorname{sech}^2\left( \frac{\mu B}{k_B T} \right) $$

Verifiquemos que $C_B(T)$ es una función par respecto a $T$ (o $1/T$), presentando el característico **pico de Schottky** tanto para $T > 0$ como para $T < 0$, y convergiendo a cero cuando $T \to \pm 0\,\text{K}$ (conforme al Tercer Principio de la Termodinámica).

In [ ]:
# Cálculo y visualización de la capacidad calorífica C_B(T) y C_B(beta)
beta_muB_vals = np.linspace(-4.0, 4.0, 500)

# Capacidad calorífica adimensional C_B / (N k_B) = (beta * muB)^2 / cosh^2(beta * muB)
C_per_spin = (beta_muB_vals ** 2) / (np.cosh(beta_muB_vals) ** 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: C_B vs beta (escala continua natural)
ax1.plot(beta_muB_vals, C_per_spin, color="crimson", linewidth=2.5, label=r"$C_B / (N k_B) = (\beta\mu B)^2 \operatorname{sech}^2(\beta\mu B)$")
ax1.axvline(0, color="black", linestyle=":", alpha=0.5, label=r"$\beta = 0$ ($T = \pm \infty, C_B = 0$)")
ax1.set_title(r"Capacidad Calorífica vs. Parámetro Inverso $\beta$")
ax1.set_xlabel(r"$\beta \,\mu B$")
ax1.set_ylabel(r"$C_B / (N k_B)$")
ax1.legend()

# Panel 2: C_B vs Temperatura T (mostrando ramas T > 0 y T < 0)
T_pos = np.linspace(0.05, 5.0, 300)
T_neg = np.linspace(-5.0, -0.05, 300)

C_pos = (1.0 / T_pos)**2 / (np.cosh(1.0 / T_pos)**2)
C_neg = (1.0 / T_neg)**2 / (np.cosh(1.0 / T_neg)**2)

ax2.plot(T_pos, C_pos, "b-", linewidth=2.2, label=r"Rama $T > 0$ (Pico de Schottky en $k_B T/\mu B \approx 0.83$)")
ax2.plot(T_neg, C_neg, "r-", linewidth=2.2, label=r"Rama $T < 0$ (Simétrica)")
ax2.axvline(0, color="black", linestyle="--", alpha=0.6)
ax2.set_title(r"Anomalía de Schottky: $C_B(T)$ en $T > 0$ y $T < 0$")
ax2.set_xlabel(r"Temperatura $k_B T / (\mu B)$")
ax2.set_ylabel(r"$C_B / (N k_B)$")
ax2.legend()

plt.tight_layout()
plt.show()

### 🧠 Reto 2: Muestreo Microcanónico de Configuraciones de Espines
Visualicemos directamente la estructura microscópica de una red de $N = 20 \times 20 = 400$ espines fijando la energía total en dos regímenes opuestos:
1. **Régimen de Temperatura Positiva ($T > 0$, $\varepsilon = -0.7$):** predominio masivo de espines arriba (azules).
2. **Régimen de Temperatura Negativa ($T < 0$, $\varepsilon = +0.7$):** inversión de población con predominio de espines abajo (rojos).

In [ ]:
# Visualización de microestados instantáneos en la red 2D para T > 0 vs T < 0
L_grid = 20
N_total = L_grid * L_grid
rng = np.random.default_rng(42)

def generate_microstate(eps_target, N_tot, L, random_state):
    n_plus = int(round((1 - eps_target) * N_tot / 2.0))
    spins = np.ones(N_tot, dtype=int)
    indices_minus = random_state.choice(N_tot, size=N_tot - n_plus, replace=False)
    spins[indices_minus] = -1
    return spins.reshape((L, L))

grid_T_pos = generate_microstate(eps_target=-0.70, N_tot=N_total, L=L_grid, random_state=rng)
grid_T_neg = generate_microstate(eps_target=+0.70, N_tot=N_total, L=L_grid, random_state=rng)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

# Gráfica T > 0
im1 = ax1.imshow(grid_T_pos, cmap="bwr", vmin=-1, vmax=1)
ax1.set_title(r"Microestado con $T > 0$ ($\varepsilon = -0.70$)" + "\n" + r"Predominio de espines $\uparrow$ (Azul, Estado Base)")
ax1.set_xticks([])
ax1.set_yticks([])

# Gráfica T < 0
im2 = ax2.imshow(grid_T_neg, cmap="bwr", vmin=-1, vmax=1)
ax2.set_title(r"Microestado con $T < 0$ ($\varepsilon = +0.70$)" + "\n" + r"Inversión: Predominio de espines $\downarrow$ (Rojo, Estado Excitado)")
ax2.set_xticks([])
ax2.set_yticks([])

cbar = fig.colorbar(im2, ax=[ax1, ax2], orientation="horizontal", fraction=0.05, pad=0.08, ticks=[-1, 1])
cbar.ax.set_xticklabels([r"Espín Abajo $\downarrow$ ($\epsilon = +\mu B$)", r"Espín Arriba $\uparrow$ ($\epsilon = -\mu B$)"])

plt.show()

---
## 📌 Conclusiones y Síntesis Conceptual

1. **Ensamble Microcanónico:** La base de toda la mecánica estadística descansa sobre el postulado de que todos los microestados compatibles con una energía fija $E$ tienen probabilidades *a priori* idénticas ($P_r = 1/\Omega(E)$).
2. **Origen de la Temperatura y Entropía:** La interacción térmica débil maximiza la multiplicidad conjunta $\Omega^{(0)}(E) = \Omega(E)\,\Omega'(E^{(0)} - E)$, lo que lleva naturalmente a la definición de entropía $S = k_B \ln \Omega$ y temperatura $1/T = k_B \beta = \partial S / \partial E$.
3. **Asimetría y Límite Termodinámico:** En sistemas de pocos grados de libertad, $P(E)$ es sesgada (*skewed*), pero en el límite termodinámico ($N \to \infty$), el coeficiente de asimetría $\gamma_1 \propto 1/\sqrt{N} \to 0$ y las fluctuaciones relativas $\sigma_E / E \propto 1/\sqrt{N} \to 0$ convierten a las magnitudes macroscópicas en valores deterministas.
4. **Temperaturas Absolutas Negativas ($T < 0$):** En sistemas con espectro acotado (como dipolos magnéticos de espín), la región de alta energía tiene menos estados accesibles ($dS/dE < 0$). Esto produce $\beta < 0$ y $T < 0$, asociado a una **inversión de población**.
5. **Dirección Espontánea del Calor:** La escala monotónica de calor es $\beta$. Los estados con $T < 0$ ($\beta < 0$) ceden calor espontáneamente a cualquier estado con $T > 0$ ($\beta > 0$), demostrando rigurosamente que **los estados con temperatura negativa son estrictamente más calientes** que los estados ordinarios.